## Loading Dataset

In [1]:
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/kaggle/input/dailydelhiclimatetrain-csv/DailyDelhiClimateTrain.csv')
df.head()

,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.000000,84.500000,0.000000,1015.666667
1,2013-01-02,7.400000,92.000000,2.980000,1017.800000
2,2013-01-03,7.166667,87.000000,4.633333,1018.666667
3,2013-01-04,8.666667,71.333333,1.233333,1017.166667
4,2013-01-05,6.000000,86.833333,3.700000,1016.500000


## Transforming data into univariate time series problem

In [2]:
data = pd.DataFrame(df['meantemp'], columns=['meantemp'])
data.index = pd.to_datetime(df['date'])
data.sort_index(inplace=True)
data

,meantemp
date,
2013-01-01,10.000000
2013-01-02,7.400000
2013-01-03,7.166667
2013-01-04,8.666667
2013-01-05,6.000000
...,...
2016-12-28,17.217391
2016-12-29,15.238095
2016-12-30,14.095238


In [3]:
import plotly.express as px
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.io as pio
pio.renderers.default = "kaggle"


fig = px.line(data, x=data.index, y='meantemp', labels={'value':'Temperature', 'index':'Date'})
fig.update_layout(width=1200, plot_bgcolor='skyblue',paper_bgcolor='lightblue',  title_text='Cahnging of Mean-Temp with Date-time', 
                  title_x=0.5, title_font_size=24, title_font_color='darkblue')
fig.update_traces(line={'color':'blue', 'width':1})
fig.show()

## Create sequences by overlapping 

In [4]:
import numpy as np

n = 10
seq, target = [], []
for i in range(n, len(data)):
    seq.append(data.iloc[i-n:i, 0])
    target.append(data.iloc[i, 0])

seq = np.array(seq)
target = np.array(target).reshape(-1,1)

seq.shape, target.shape

((1452, 10), (1452, 1))

## Split data into Train/Test

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(seq, target, test_size=0.3, random_state=42, shuffle=False)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1016, 10), (436, 10), (1016, 1), (436, 1))

## Model Evalution

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score

models = {
    "linear regression": LinearRegression(),
    "support vector regression": SVR(kernel='rbf', C=100, gamma='auto', epsilon=0.2),
    "random forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "decision tree": DecisionTreeRegressor(max_depth=10, max_leaf_nodes=4, random_state=42),
    "xgboost": XGBRegressor(),
    "catboost": CatBoostRegressor(verbose=0)
}

predictions_ = []
Final_report = []
for model_name, model in tqdm(models.items(), desc="Training models"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions_.append(y_pred.reshape(-1,1))

    metrics = {
        'Model': model_name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSLE': np.sqrt(mean_squared_log_error(y_test, y_pred)),
        'R2': f'{r2_score(y_test, y_pred)*100:0.2f}%' 
    }
    
    Final_report.append(metrics)


Final_report = pd.DataFrame(Final_report).set_index('Model',drop=True)
Arranged_report = Final_report.sort_values(by=['MSE'], ascending=True)
print("\nModels Performance Comparison:")
Arranged_report

Training models: 100%|██████████| 6/6 [00:02<00:00,  2.03it/s]


Models Performance Comparison:


,MAE,MSE,RMSLE,R2
Model,,,,
linear regression,1.158948,2.363200,0.066266,95.51%
random forest,1.232576,2.616815,0.069890,95.03%
catboost,1.235719,2.686477,0.069149,94.90%
xgboost,1.304450,2.923276,0.072917,94.45%
decision tree,1.983955,6.415901,0.098745,87.82%
support vector regression,2.938282,15.863055,0.169249,69.89%


In [7]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

metrics = ['MAE', 'MSE', 'RMSLE', 'R2']
fig = make_subplots(rows=2, cols=2, subplot_titles=metrics)
for i, metric in enumerate(metrics):
    fig.add_trace(
        go.Bar(x=Final_report.index, y=Final_report[metric], name=metric),
        row=i//2 +1, col=i%2 +1
    )

fig.update_layout(height=800, width=1200, title_text="Models Performance Subplots",
                 title_x=0.5, title_font_size=24, title_font_color='black')
fig.show()

In [8]:
test_indeces = data.index[len(data) - len(y_test):]
fig = make_subplots(rows=2, cols=3, subplot_titles=Final_report.index)
for i, (model_name, predict) in enumerate(zip(models.keys(), predictions_)):
    fig.add_trace(
        go.Scatter(x=test_indeces, y=np.ravel(y_test), mode='lines', name='Actual', line=dict(color='red'), showlegend=(i==0)),
        row=i//3 +1, col=i%3 +1
    )
    fig.add_trace(
        go.Scatter(x=test_indeces, y=np.ravel(predict), mode='lines', name='Prediction', line=dict(color='blue'), showlegend=(i==0)),
        row=i//3 +1, col=i%3 +1
    )

fig.update_layout(height=800, width=1200, title_text="Models Performance Subplots",
                  title_x=0.5, title_font_size=24, title_font_color='darkblue', showlegend=True)
fig.show()

### Support Vector Regression and Decision Tree algorithms have an overfitting on testing data!